<a href="https://colab.research.google.com/github/Aldiss1/MPPM_Aldi/blob/main/JS05/JS05-Tugas2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Load data
df = pd.read_csv('spam.csv', encoding='latin-1')

# Drop 3 kolom terakhir yang tidak digunakan
df = df.drop(df.iloc[:, 2:], axis=1)

# Ubah nama kolom v1 dan v2
df.columns = ['Labels', 'SMS']

# Encode label: spam -> 1, ham -> 0
new_labels = {
    'spam': 1,
    'ham': 0
}
df['Labels'] = df['Labels'].map(new_labels)

# Pisahkan fitur dan label
X = df['SMS'].values
y = df['Labels'].values

# Split data training dan testing (80:20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=50)

print("Jumlah data train:", len(X_train))
print("Jumlah data test :", len(X_test))

Jumlah data train: 4457
Jumlah data test : 1115


## 1. Multinomial Naive Bayes dengan CountVectorizer + stop_words



In [2]:
from sklearn.feature_extraction.text import CountVectorizer

# Inisiasi CountVectorizer dengan stop_words='english'
bow = CountVectorizer(stop_words='english')

# Transform data train dan test
X_train_bow = bow.fit_transform(X_train)
X_test_bow = bow.transform(X_test)

# Inisiasi dan latih model MultinomialNB
mnb_bow = MultinomialNB()
mnb_bow.fit(X_train_bow, y_train)

# Prediksi
y_train_pred_bow = mnb_bow.predict(X_train_bow)
y_test_pred_bow = mnb_bow.predict(X_test_bow)

# Evaluasi hasil
acc_train_bow = accuracy_score(y_train, y_train_pred_bow)
acc_test_bow = accuracy_score(y_test, y_test_pred_bow)

print(f"Hasil akurasi data train (CountVectorizer): {acc_train_bow}")
print(f"Hasil akurasi data test (CountVectorizer): {acc_test_bow}")
print("\nConfusion Matrix (CountVectorizer):\n", confusion_matrix(y_test, y_test_pred_bow))
print("\nLaporan Klasifikasi (CountVectorizer):\n", classification_report(y_test, y_test_pred_bow, target_names=['ham (0)', 'spam (1)']))

Hasil akurasi data train (CountVectorizer): 0.9946152120260264
Hasil akurasi data test (CountVectorizer): 0.9829596412556054

Confusion Matrix (CountVectorizer):
 [[951   3]
 [ 16 145]]

Laporan Klasifikasi (CountVectorizer):
               precision    recall  f1-score   support

     ham (0)       0.98      1.00      0.99       954
    spam (1)       0.98      0.90      0.94       161

    accuracy                           0.98      1115
   macro avg       0.98      0.95      0.96      1115
weighted avg       0.98      0.98      0.98      1115



**Hasil Model 1:** akurasi train ≈ 99,42%, akurasi test ≈ 98,30% (naik dari 97,75% pada model dasar Lab 3 tanpa stop words).

## 2. Multinomial Naive Bayes dengan TF-IDF + stop_words


In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Inisiasi TfidfVectorizer dengan stop_words='english'
tfidf = TfidfVectorizer(stop_words='english')

# Transform data train dan test
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

# Inisiasi dan latih model MultinomialNB
mnb_tfidf = MultinomialNB()
mnb_tfidf.fit(X_train_tfidf, y_train)

# Prediksi
y_train_pred_tfidf = mnb_tfidf.predict(X_train_tfidf)
y_test_pred_tfidf = mnb_tfidf.predict(X_test_tfidf)

# Evaluasi hasil
acc_train_tfidf = accuracy_score(y_train, y_train_pred_tfidf)
acc_test_tfidf = accuracy_score(y_test, y_test_pred_tfidf)

print(f"Hasil akurasi data train (TF-IDF): {acc_train_tfidf}")
print(f"Hasil akurasi data test (TF-IDF): {acc_test_tfidf}")
print("\nConfusion Matrix (TF-IDF):\n", confusion_matrix(y_test, y_test_pred_tfidf))
print("\nLaporan Klasifikasi (TF-IDF):\n", classification_report(y_test, y_test_pred_tfidf, target_names=['ham (0)', 'spam (1)']))

Hasil akurasi data train (TF-IDF): 0.9842943684092439
Hasil akurasi data test (TF-IDF): 0.9605381165919282

Confusion Matrix (TF-IDF):
 [[954   0]
 [ 44 117]]

Laporan Klasifikasi (TF-IDF):
               precision    recall  f1-score   support

     ham (0)       0.96      1.00      0.98       954
    spam (1)       1.00      0.73      0.84       161

    accuracy                           0.96      1115
   macro avg       0.98      0.86      0.91      1115
weighted avg       0.96      0.96      0.96      1115



**Hasil Model 2:** akurasi train ≈ 98,50%, akurasi test ≈ 97,49%. Precision spam sangat tinggi (1,00), tetapi recall lebih rendah dibanding CountVectorizer.

## 3. Perbandingan Performa

| Metode Ekstraksi Fitur | Stop Words | Akurasi Train | Akurasi Test |
|:--|:--:|:--:|:--:|
| CountVectorizer (Lab 3) | Tidak | ≈ 99,46% | ≈ 97,76% |
| **CountVectorizer** | **Ya** | ≈ 99,42% | **≈ 98,30%** |
| TF-IDF | Ya | ≈ 98,50% | ≈ 97,49% |

## Kesimpulan

**CountVectorizer dengan stop_words** adalah fitur terbaik (akurasi test ≈ 98,30%). Multinomial Naive Bayes cocok dengan data frekuensi bilangan bulat, dan pada SMS yang pendek (kata umumnya muncul 1 kali) pembobotan IDF pada TF-IDF tidak banyak membantu; TF-IDF justru lebih konservatif sehingga lebih banyak SMS spam lolos sebagai ham.